In [ ]:
#pip install librosa matplotlib scipy soundfile noisereduce

In [ ]:
import soundfile as sf

y, sr = sf.read("concert.wav")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import spectrogram

y_mono = np.mean(y, axis=1) if y.ndim > 1 else y

frequencies, times, spectrum = spectrogram(
    y_mono,
    fs=sr,
    nperseg=2048,
    noverlap=1024
)
spectrum_db = 10 * np.log10(spectrum + 1e-12)

plt.figure(figsize=(12, 5))
plt.pcolormesh(
    times,
    frequencies,
    spectrum_db,
    shading="auto",
    cmap="magma"
)

plt.colorbar(format="%+2.0f dB")
plt.title("Original Spectrum")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.ylim(0, 8000)
plt.tight_layout()
plt.show()

In [ ]:
#pip install noisereduce

In [ ]:
import numpy as np
import noisereduce as nr

noise_reduction_amount = 0.35

y_for_denoise = y.T if y.ndim > 1 else y

reduced_for_denoise = nr.reduce_noise(
    y=y_for_denoise,
    sr=sr,
    stationary=False,
    prop_decrease=noise_reduction_amount,
    time_constant_s=4.0,
    freq_mask_smooth_hz=1200,
    time_mask_smooth_ms=200,
    thresh_n_mult_nonstationary=1.5,
    sigmoid_slope_nonstationary=8,
    n_fft=2048,
    hop_length=512,
    use_tqdm=False
)

reduced = reduced_for_denoise.T if y.ndim > 1 else reduced_for_denoise
reduced = np.nan_to_num(reduced)

print("Music-first noise reduction complete!")

In [ ]:
import soundfile as sf

sf.write(
    "clean_music_first.wav",
    reduced,
    sr
)

print("Wrote clean_music_first.wav")

In [ ]:
from scipy.signal import butter,filtfilt

In [ ]:
import numpy as np
import soundfile as sf
from scipy.signal import butter, filtfilt


def highpass_filter(data, cutoff, fs, order=4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist

    b, a = butter(order, normal_cutoff, btype='high')
    return filtfilt(b, a, data)


def bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)


def vocal_enhancement(audio, sr):

    cleaned = highpass_filter(audio, 60, sr)

    vocal_band = bandpass_filter(
        cleaned,
        lowcut=1000,
        highcut=4000,
        fs=sr
    )

    enhanced = cleaned + 0.25 * vocal_band

    max_amp = np.max(np.abs(enhanced))
    if max_amp > 0:
        enhanced = enhanced / max_amp

    return enhanced


if __name__ == "__main__":

    input_path = "clean_music_first.wav"
    output_path = "enhanced_music_first.wav"

    audio, sr = sf.read(input_path)

    if audio.ndim > 1:
        enhanced = np.column_stack([
            vocal_enhancement(audio[:, channel], sr)
            for channel in range(audio.shape[1])
        ])
    else:
        enhanced = vocal_enhancement(audio, sr)

    sf.write(
        output_path,
        enhanced,
        sr
    )

    print("EQ enhancement complete!")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from scipy.signal import spectrogram


clean_path = Path("clean_music_first.wav")
enhanced_path = Path("enhanced_music_first.wav")

for audio_path in (clean_path, enhanced_path):
    if not audio_path.exists():
        raise FileNotFoundError(f"Run the previous processing cells first to create {audio_path}")

enhanced_audio, enhanced_sr = sf.read(enhanced_path)

if enhanced_audio.ndim > 1:
    enhanced_audio = np.mean(enhanced_audio, axis=1)

frequencies, times, enhanced_spectrum = spectrogram(
    enhanced_audio,
    fs=enhanced_sr,
    nperseg=2048,
    noverlap=1024
)
enhanced_spectrum_db = 10 * np.log10(enhanced_spectrum + 1e-12)

plt.figure(figsize=(12, 5))
plt.pcolormesh(
    times,
    frequencies,
    enhanced_spectrum_db,
    shading="auto",
    cmap="magma"
)
plt.colorbar(format="%+2.0f dB")
plt.title("Music-First EQ Enhanced Spectrum")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.ylim(0, 8000)
plt.tight_layout()
plt.show()

print("Music-first denoised:")
display(Audio(filename=str(clean_path)))

print("Music-first + EQ enhanced:")
display(Audio(filename=str(enhanced_path)))

In [ ]:
from pathlib import Path

from IPython.display import Audio, display


comparison_files = [
    ("Original", "concert.wav"),
    ("Mild denoise - prop_decrease 0.35", "clean_music_first.wav"),
    ("Medium denoise - prop_decrease 0.55", "clean_music_medium.wav"),
    ("Stronger denoise - prop_decrease 0.70", "clean_music_stronger.wav"),
    ("Mild + EQ", "enhanced_music_first.wav"),
    ("Medium + EQ", "enhanced_music_medium.wav"),
    ("Stronger + EQ", "enhanced_music_stronger.wav"),
]

for label, file_name in comparison_files:
    path = Path(file_name)
    if path.exists():
        print(label)
        display(Audio(filename=str(path)))
    else:
        print(f"Missing {file_name}")